# MISION: Deep Learning para Secuencias

**Objetivo:** Usar una LSTM (Long Short-Term Memory) para predecir RUL aprovechando la naturaleza **secuencial** de los datos.

---

### Por que LSTM?

Los modelos clasicos (Random Forest, XGBoost) tratan cada observacion como independiente. Pero un motor degrada **progresivamente** — el ciclo 100 depende de lo que paso en los ciclos 95-99. Una LSTM puede:

- Capturar patrones temporales de degradacion
- Recordar estados anteriores del motor
- Detectar transiciones sutiles entre "sano" y "degradado"

**Arquitectura:** LSTM(64) -> Dropout(0.2) -> LSTM(32) -> Dropout(0.2) -> Dense(1)

## 1. Imports y Configuracion

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import warnings
warnings.filterwarnings('ignore')

import tensorflow as tf
from tensorflow import keras
from keras import layers, callbacks

print(f"TensorFlow: {tf.__version__}")
print(f"GPU disponible: {len(tf.config.list_physical_devices('GPU')) > 0}")

plt.style.use('seaborn-v0_8-whitegrid')
HEALTHY = '#2ecc71'
DEGRADED = '#f39c12'
CRITICAL = '#e74c3c'
NEUTRAL = '#3498db'

import os
project_root = os.path.abspath(os.path.join(os.getcwd(), "../../.."))
processed_dir = os.path.join(project_root, "data", "cmapss", "processed")
models_dir = os.path.join(project_root, "models")

np.random.seed(42)
tf.random.set_seed(42)

2026-03-21 20:35:50.588547: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-21 20:35:50.598705: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:479] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-03-21 20:35:50.614251: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:10575] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-03-21 20:35:50.614280: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1442] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-03-21 20:35:50.623775: I tensorflow/core/platform/cpu_feature_gua

TensorFlow: 2.16.2
GPU disponible: True


2026-03-21 20:35:51.700188: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:998] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2026-03-21 20:35:51.730619: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:998] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2026-03-21 20:35:51.730754: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:998] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-

## 2. Cargar Datos y Preparar Secuencias

La LSTM espera datos en formato `(samples, timesteps, features)`. Creamos ventanas deslizantes de longitud `window_size` para cada motor.

In [2]:
df_train = pd.read_parquet(os.path.join(processed_dir, 'train_features.parquet'))
df_test = pd.read_parquet(os.path.join(processed_dir, 'test_features.parquet'))

feature_cols = [c for c in df_train.columns if c not in ['unit', 'time_cycles', 'RUL']]
print(f"Train: {df_train.shape}")
print(f"Test: {df_test.shape}")
print(f"Features: {len(feature_cols)}")

Train: (20631, 61)
Test: (13096, 61)
Features: 58


In [3]:
WINDOW_SIZE = 30

def create_sequences(df, feature_cols, window_size):
    """
    Crear ventanas temporales para LSTM.
    Para cada motor, genera secuencias de longitud window_size.
    Si el motor tiene menos ciclos que window_size, se usa padding con el primer valor.
    """
    X, y = [], []
    
    for unit in df['unit'].unique():
        unit_data = df[df['unit'] == unit].sort_values('time_cycles')
        features = unit_data[feature_cols].values
        rul = unit_data['RUL'].values
        
        # Padding si el motor tiene menos ciclos que window_size
        if len(features) < window_size:
            pad_length = window_size - len(features)
            features = np.vstack([np.tile(features[0], (pad_length, 1)), features])
            rul = np.concatenate([np.full(pad_length, rul[0]), rul])
        
        # Crear ventanas deslizantes
        for i in range(window_size, len(features) + 1):
            X.append(features[i - window_size:i])
            y.append(rul[i - 1])
    
    return np.array(X), np.array(y)

X_train_seq, y_train_seq = create_sequences(df_train, feature_cols, WINDOW_SIZE)
X_test_seq, y_test_seq = create_sequences(df_test, feature_cols, WINDOW_SIZE)

print(f"X_train: {X_train_seq.shape} (samples, timesteps, features)")
print(f"y_train: {y_train_seq.shape}")
print(f"X_test: {X_test_seq.shape}")
print(f"y_test: {y_test_seq.shape}")

X_train: (17731, 30, 58) (samples, timesteps, features)
y_train: (17731,)
X_test: (10196, 30, 58)
y_test: (10196,)


## 3. Test: Solo Ultimo Ciclo

Para comparar con los modelos clasicos, evaluamos la LSTM en el **ultimo ciclo** de cada motor del test set.

In [4]:
# Extraer solo las secuencias del ultimo ciclo de cada motor en test
def get_last_sequences(df, feature_cols, window_size):
    """Obtener la secuencia del ultimo ciclo de cada motor."""
    X_last, y_last, units = [], [], []
    
    for unit in df['unit'].unique():
        unit_data = df[df['unit'] == unit].sort_values('time_cycles')
        features = unit_data[feature_cols].values
        rul = unit_data['RUL'].values
        
        if len(features) < window_size:
            pad_length = window_size - len(features)
            features = np.vstack([np.tile(features[0], (pad_length, 1)), features])
            rul = np.concatenate([np.full(pad_length, rul[0]), rul])
        
        X_last.append(features[-window_size:])
        y_last.append(rul[-1])
        units.append(unit)
    
    return np.array(X_last), np.array(y_last), units

X_test_last, y_test_last, test_units = get_last_sequences(df_test, feature_cols, WINDOW_SIZE)
print(f"X_test (ultimo ciclo): {X_test_last.shape}")
print(f"y_test (ultimo ciclo): {y_test_last.shape}")

X_test (ultimo ciclo): (100, 30, 58)
y_test (ultimo ciclo): (100,)


## 4. Arquitectura LSTM

Dos capas LSTM con dropout para regularizacion. La capa Dense final produce una unica salida: el RUL predicho.

In [5]:
def build_lstm_model(input_shape):
    model = keras.Sequential([
        layers.LSTM(64, return_sequences=True, input_shape=input_shape),
        layers.Dropout(0.2),
        layers.LSTM(32, return_sequences=False),
        layers.Dropout(0.2),
        layers.Dense(16, activation='relu'),
        layers.Dense(1)
    ])
    
    model.compile(optimizer=keras.optimizers.Adam(learning_rate=0.001),
                  loss='mse',
                  metrics=['mae'])
    return model

model = build_lstm_model((WINDOW_SIZE, len(feature_cols)))
model.summary()

2026-03-21 20:35:52.008966: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:998] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2026-03-21 20:35:52.009186: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:998] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2026-03-21 20:35:52.009350: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:998] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 30, 64)         │        31,488 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 30, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 32)             │        12,416 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 44,449 (173.63 KB)

 Trainable params: 44,449 (173.63 KB)

 Non-trainable params: 0 (0.00 B)

## 5. Entrenamiento

In [ ]:
early_stop = callbacks.EarlyStopping(monitor='val_loss', patience=10, 
                                      restore_best_weights=True, verbose=1)
reduce_lr = callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, 
                                         patience=5, min_lr=1e-6, verbose=1)

history = model.fit(
    X_train_seq, y_train_seq,
    epochs=50,
    batch_size=256,
    validation_split=0.2,
    callbacks=[early_stop, reduce_lr],
    verbose=1
)

Epoch 1/50


2026-03-21 20:35:54.457278: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:465] Loaded cuDNN version 8907


56/56 ━━━━━━━━━━━━━━━━━━━━ 3s 14ms/step - loss: 7479.1157 - mae: 75.1622 - val_loss: 7380.6499 - val_mae: 74.6917 - learning_rate: 0.0010
Epoch 2/50
56/56 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - loss: 5882.5825 - mae: 65.2665 - val_loss: 5759.9058 - val_mae: 65.1226 - learning_rate: 0.0010
Epoch 3/50
56/56 ━━━━━━━━━━━━━━━━━━━━ 1s 18ms/step - loss: 4474.1182 - mae: 56.3940 - val_loss: 4327.8970 - val_mae: 56.4773 - learning_rate: 0.0010
Epoch 4/50
56/56 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - loss: 3321.6584 - mae: 48.9426 - val_loss: 3212.4062 - val_mae: 49.4212 - learning_rate: 0.0010
Epoch 5/50
56/56 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - loss: 2528.2166 - mae: 43.5725 - val_loss: 2509.2488 - val_mae: 44.6095 - learning_rate: 0.0010
Epoch 6/50
56/56 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - loss: 2121.8018 - mae: 40.6067 - val_loss: 2162.4600 - val_mae: 41.9176 - learning_rate: 0.0010
Epoch 7/50
56/56 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - loss: 1961.6118 - mae: 39.2228 - val_loss: 2025.6655 - val_mae: 

In [ ]:
# Curvas de entrenamiento
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

ax = axes[0]
ax.plot(history.history['loss'], label='Train Loss', color=NEUTRAL, linewidth=2)
ax.plot(history.history['val_loss'], label='Val Loss', color=CRITICAL, linewidth=2)
ax.set_title('Loss (MSE)', fontsize=13, fontweight='bold')
ax.set_xlabel('Epoch')
ax.set_ylabel('MSE')
ax.legend()

ax = axes[1]
ax.plot(history.history['mae'], label='Train MAE', color=NEUTRAL, linewidth=2)
ax.plot(history.history['val_mae'], label='Val MAE', color=CRITICAL, linewidth=2)
ax.set_title('MAE', fontsize=13, fontweight='bold')
ax.set_xlabel('Epoch')
ax.set_ylabel('MAE (ciclos)')
ax.legend()

plt.suptitle('Curvas de Entrenamiento LSTM', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 6. Evaluacion

In [ ]:
def nasa_score(y_true, y_pred):
    """NASA Scoring Function."""
    d = y_pred - y_true
    scores = np.where(d < 0, np.exp(-d / 13) - 1, np.exp(d / 10) - 1)
    return np.sum(scores)

# Predecir en ultimo ciclo de cada motor (comparable con modelos clasicos)
y_pred_lstm = model.predict(X_test_last, verbose=0).flatten()

rmse = np.sqrt(mean_squared_error(y_test_last, y_pred_lstm))
mae = mean_absolute_error(y_test_last, y_pred_lstm)
r2 = r2_score(y_test_last, y_pred_lstm)
score = nasa_score(y_test_last, y_pred_lstm)

print("=" * 50)
print("  RESULTADOS LSTM")
print("=" * 50)
print(f"  RMSE:       {rmse:.2f} ciclos")
print(f"  MAE:        {mae:.2f} ciclos")
print(f"  R2:         {r2:.4f}")
print(f"  NASA Score: {score:,.0f}")

## 7. Comparacion con Modelos Clasicos

In [ ]:
import joblib

# Cargar resultados de modelos clasicos
metadata = joblib.load(os.path.join(models_dir, 'rul_predictor_metadata.pkl'))
classic_results = metadata['results']

# Agregar LSTM
all_results = classic_results + [{'model': 'LSTM', 'RMSE': rmse, 'MAE': mae, 'R2': r2, 'NASA_Score': score}]
results_df = pd.DataFrame(all_results).sort_values('RMSE')

print("COMPARACION FINAL: TODOS LOS MODELOS")
print("=" * 75)
for _, row in results_df.iterrows():
    marker = " <-- MEJOR" if row['RMSE'] == results_df['RMSE'].min() else ""
    print(f"  {row['model']:<25} RMSE={row['RMSE']:6.2f}  MAE={row['MAE']:6.2f}  "
          f"R2={row['R2']:.4f}  NASA={row['NASA_Score']:>8,.0f}{marker}")
print("=" * 75)

results_df

## 8. Visualizacion: Prediccion vs Real para Motores del Test

In [ ]:
# Predecir RUL completo para 5 motores del test
sample_test_units = sorted(df_test['unit'].unique())[:5]

fig, axes = plt.subplots(len(sample_test_units), 1, figsize=(16, 4 * len(sample_test_units)))

for idx, unit in enumerate(sample_test_units):
    ax = axes[idx]
    unit_data = df_test[df_test['unit'] == unit].sort_values('time_cycles')
    
    # Crear secuencias para este motor
    features = unit_data[feature_cols].values
    rul_real = unit_data['RUL'].values
    cycles = unit_data['time_cycles'].values
    
    # Padding si necesario
    if len(features) < WINDOW_SIZE:
        pad_length = WINDOW_SIZE - len(features)
        features_padded = np.vstack([np.tile(features[0], (pad_length, 1)), features])
    else:
        features_padded = features
    
    # Predecir para cada ciclo
    preds = []
    for i in range(WINDOW_SIZE, len(features_padded) + 1):
        seq = features_padded[i - WINDOW_SIZE:i].reshape(1, WINDOW_SIZE, -1)
        pred = model.predict(seq, verbose=0)[0, 0]
        preds.append(pred)
    
    # Ajustar longitud
    preds = preds[-len(cycles):]
    
    ax.plot(cycles, rul_real, color=CRITICAL, linewidth=2, label='RUL Real')
    ax.plot(cycles, preds, color=NEUTRAL, linewidth=2, linestyle='--', label='RUL Predicho (LSTM)')
    ax.fill_between(cycles, rul_real, preds, alpha=0.2, color=DEGRADED)
    ax.set_title(f'Motor {unit}', fontsize=12, fontweight='bold')
    ax.set_xlabel('Ciclo')
    ax.set_ylabel('RUL')
    ax.legend()

plt.suptitle('LSTM: Prediccion vs RUL Real (Motores del Test)', fontsize=15, fontweight='bold', y=1.001)
plt.tight_layout()
plt.show()

## 9. Guardar Modelo

In [ ]:
model_path = os.path.join(models_dir, 'rul_lstm.keras')
model.save(model_path)
print(f"Modelo LSTM guardado: {model_path}")
print(f"Tamano: {os.path.getsize(model_path) / 1024:.1f} KB")

# Guardar resultados actualizados
joblib.dump({
    'feature_cols': metadata['feature_cols'],
    'results': all_results,
    'window_size': WINDOW_SIZE
}, os.path.join(models_dir, 'rul_predictor_metadata.pkl'))
print("Metadata actualizada con resultados LSTM")

## Resumen

1. **Arquitectura:** LSTM(64) -> Dropout -> LSTM(32) -> Dropout -> Dense(16) -> Dense(1)
2. **Ventana temporal:** 30 ciclos
3. **Ventaja LSTM:** Captura patrones secuenciales que los modelos clasicos ignoran
4. **Comparacion:** La LSTM deberia mejorar el RMSE y NASA Score respecto a XGBoost/RF

### Siguiente paso
Evaluacion de produccion: NASA Score detallado, sistema de alertas, simulacion operativa.